In [389]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import stackview as st
import iplotx as ipx
import numpy as np
import glob
import json
import skimage as ski
import scipy.ndimage as ndi
import scipy.signal as sig
import itertools

from tifffile import imread
from skimage.color import rgb2hsv

from pyfibre.fibers.network_extraction import *
from pyfibre.fibers.metrics import *
from pyfibre.tools.utilities import *
from pyfibre.tools.convertors import *
from pyfibre.fibers.convertors import *


In [130]:
params = {
    "scale": 0.5,
    "r_thresh": 25,
    "nuc_radius": 15,
    "sigma": 0.5,
    "angle_thresh": 70
}

In [448]:
stack = imread('/home/vpc/A3_NDTiffStack.tif')
img = stack[3]
img_nonpol = stack[2]
st.blend(img, img_nonpol, zoom_factor=0.25)

In [5]:
def mask_hue(
    h,
    sigma_hue=2,
    hue_range=(0, 0.5),
    ):
    """
    Gets rid of blue hued pixels in the polarized image.
    Blue sits on the high end of the HSV hue channel,
    so just exclude it completely by only getting 0-0.5
    """

    h_smoothed = ski.exposure.rescale_intensity(
        ndi.gaussian_filter(h, sigma=sigma_hue),
        in_range=hue_range
    )

    h_mask = h_smoothed < ski.filters.threshold_otsu(h_smoothed)
    h_mask = ski.morphology.remove_small_objects(h_mask)

    return h_mask


In [17]:
def denoise(
    img,
    p_range=(1, 99),
    sigma=2
    ):
    """
    Denoise image using denoise_nl_means
    """

    params["sigma"] = ski.restoration.estimate_sigma(img)

    p1, p2 = np.percentile(img, p_range)

    img_dn = ndi.gaussian_filter(
        ski.exposure.equalize_adapthist(
            ski.exposure.rescale_intensity(
                img,
                in_range=(p1, p2),
                out_range=(p1, p2)
            )
        ),
        sigma=sigma
    )

    return img_dn

In [360]:
def make_tissue_mask(
    img,
    sigma=1,
    scale=0.5,
    ent_disk_r=(5,3),
    ent_nbins=32,
    ent_prominence=0.1,
    ent_dist=15,
    ):
    img = ski.transform.rescale(
        img,
        scale,
        mode='constant',
        anti_aliasing=None,
        channel_axis=-1
    )

    _, s, v = np.unstack(ski.color.rgb2hsv(img), axis=-1)

    img_gray = ndi.gaussian_filter(
        v,
        sigma=sigma
    )

    ent = ski.filters.rank.entropy(
        img_gray,
        ski.morphology.disk(ent_disk_r[1])
    )

    ent_count, ent_bins = np.histogram(ent, ent_nbins)
    ent_bins = ent_bins[1:]

    min_idx = sig.argrelextrema(ent_count, np.less)
    min_idx = np.array(*min_idx)

    peak_idx, peak_props = sig.find_peaks(
        ent_count,
        prominence=ent_prominence,
        distance=ent_dist
    )

    low, high = [
        ski.measure.shannon_entropy(img_gray, base=i)
        for i in [32, 4]
    ]

    min_x, min_y = [], []

    for idx in min_idx:
        x = ent_bins[idx]
        y = ent_count[idx]

        min_x.append(x)
        min_y.append(y)

    peak_x, peak_y = [], []

    for idx in peak_idx:
        x = ent_bins[idx]
        y = ent_count[idx]

        peak_x.append(x)
        peak_y.append(y)

    for binned in min_x:
        temp_thresh = binned
        if temp_thresh > 1 and temp_thresh < 4:
            thresh_local = temp_thresh

    tissue_mask = ent > thresh_local
    tissue_mask = ndi.binary_erosion(
        tissue_mask,
        structure=ndi.iterate_structure(
            ndi.generate_binary_structure(2,1),
            iterations=2
        )
    )
    tissue_mask = ndi.binary_fill_holes(tissue_mask)

    lbl = ski.measure.label(tissue_mask)

    properties = [
        'label',
        'area',
        'major_axis_length',
        'minor_axis_length',
        'intensity_mean'
    ]

    props = ski.measure.regionprops_table(
        lbl,
        intensity_image=ndi.gaussian_filter(s, sigma=1),
        properties = properties
        )

    props_df = pd.DataFrame(props)

    lb, ar, Mal, mal, _ = [ props_df[prop] for prop in properties ]

    props_df['circularity'] = ar / ( Mal / 2 * mal / 2 * np.pi )

    blem = props_df.query("circularity > 0.9 and intensity_mean < 0.2")

    exclude = blem['label'].values

    mask = np.isin(lbl,exclude)
    lbl_clean = lbl.copy()
    lbl_clean[mask] = 0
    
    tissue_mask = lbl_clean > 0
    tissue_mask = ski.transform.rescale(
        tissue_mask,
        1 / scale,
        mode='constant',
        anti_aliasing=None
    )
    return tissue_mask

In [442]:
def analyze(
    img,
    img_nonpol
):
    h, s, v = np.unstack(rgb2hsv(img), axis=-1)
    v_ = v * mask_hue(h)
    v_dn = denoise(v_)
    net_graph = build_network(
        v_dn, **params
    )
    nets = fibre_network_assignment(net_graph)

    n_fibs = 0
    valid_nets = []
    net_graphs = []

    for net in nets:
        fibs = net.generate_fibres()
        if len(fibs) > 1:
            n_fibs += len(fibs)
            valid_nets.append(net)
            net_graphs.append(net.graph)

    n_nets = len(valid_nets)
    metrics = fibre_network_metrics(valid_nets)

    tissue_mask = make_tissue_mask(img_nonpol)
    binary = networks_to_binary(net_graphs, shape=v_dn.shape)
    h_mask = ski.morphology.remove_small_objects(
        (mask_hue(h) * binary).astype('bool')
    )

    hb = h * h_mask

    tissue_px = np.argwhere(tissue_mask)
    fib_px = np.argwhere(hb)

    red =       ((hb >= 2/255) & (hb <= 9/255)) | ((hb >= 230/255) & (hb <= 1.0))
    orange =    ((hb >=10/255)) & ((hb <= 38/255))
    yellow =    ((hb >= 39/255) & (hb <= 51/255))
    green =     ((hb >= 52/255) & (hb <= 128/255))

    colors = [red, orange, yellow, green]

    sum_all = np.sum([len(np.argwhere(color)) for color in colors])

    red_pct, orange_pct, yellow_pct, green_pct = [ 100 * len(np.argwhere(color)) / sum_all for color in colors ]

    col_pct = 100 * len(fib_px) / len(tissue_px)

    stats = pd.Series()
    stats["Fiber No."] = n_fibs
    stats["Network No."] = n_nets
    stats["Collagen area"] = col_pct
    stats["Red collagen"] = red_pct
    stats["Orange collagen"] = orange_pct
    stats["Yellow collagen"] = yellow_pct
    stats["Green collagen"] = green_pct

    return metrics, stats, valid_nets, valid_graphs

In [454]:
def plot_polar(angles, fname): 
   fig, ax = plt.subplots(subplot_kw=dict(projection="polar"))
   doubled = (2 * np.array(angles)) % (2 * np.pi)

   ax.hist(doubled, bins=36, alpha=0.85, density=True)
   ax.set_title("Fiber Orientation", pad=15, fontsize=11, fontweight="bold")
   ax.set_theta_zero_location("E")

   sin_mean = np.sin(doubled).mean()
   cos_mean = np.cos(doubled).mean()
   R = np.sqrt(sin_mean**2 + cos_mean**2)
   circ_std = np.degrees(np.sqrt(-2 * np.log(R + 1e-10))) / 2

   ax.set_xlabel(f"Resultant vector R = {R:.3f}  |  Circ. std = {circ_std:.1f}°",
                    labelpad=15, fontsize=10)

   plt.tight_layout()
   plt.savefig(fname, dpi=150)

In [455]:
def plot_fibers(img, nets, fname):
    """
    Plot overlay from image and list of Networks
    """

    fig, ax = plt.subplots()
    ax.imshow(img, cmap='gray') #v_dn

    palette = itertools.cycle([
        "red",
        "green",
        "blue",
        "orange",
        "yellow",
        "white"
    ])

    for net in nets:
        for fib in net.generate_fibres():
            subg = fib.graph
            color = next(palette)
            layout = {
                n: np.flip(subg.nodes[n]['xy']) for n in subg.nodes
            }
            ipx.network(subg,
            layout=layout,
            ax=ax,
            style=['tree', {
                "vertex": {
                    "size": 0
                },
                "edge": {
                    "linewidth": 1,
                    "color": color
                }
            }],
            alpha=0.7)

    plt.tight_layout()
    plt.savefig(fname, dpi=150)

In [ ]:
img_dir = '/run/media/vpc/kaffee/Computador_micro/PICRO-TUMOR-JULHO_2026'
out_dir = ''

for group in os.listdir(img_dir):
    files = glob.glob(f"{os.path.join(img_dir, group)}/**/*.tif", recursive=True)
    print(group)
    
    for file in files:
        name = os.path.basedir(file)
        name = name.split("_")[0]
        stack = imread(file)

        stack_metrics = pd.DataFrame()
        stack_stats = pd.DataFrame()
        fib_angles = []
        fib_lengths = []

        pol_idx = []
        nonpol_idx = []
        for n in range(0, len(stack)):
            if n == 0 or n % 2 == 0:
                nonpol_idx.append(n)
            else:
                pol_idx.append(n)

        for p, n in zip(pol_idx, nonpol_idx):
            metrics, stats, nets, graphs = analyze(
                stack[p],
                stack[n]
            )

            stack_metrics = pd.concat(
                [stack_metrics, metrics],
                ignore_index=True
            )

            stack_stats = pd.concat(
                [stack_stats, stats.to_frame().T], 
                ignore_index=True
            )

            for net in nets:
                for fib in net.generate_fibres():
                    fib_angles.append(fib.angle)
                    fib_lengths.append(fib.fiber_l)

            plot_fibers(
                stack[p], nets,
                os.path.join(out_dir, f"{name}_{n}-fibers.png")
            )

        plot_polar(
            fib_angles,
            os.path.join(out_dir, f"{name}-angles.png")
        )

Fêmeas
/run/media/vpc/kaffee/Computador_micro/PICRO-TUMOR-JULHO_2026/Fêmeas/6A/6A_NDTiffStack.tif 16
pair: 1 polarized, 0 non-polarized
pair: 3 polarized, 2 non-polarized
pair: 5 polarized, 4 non-polarized
pair: 7 polarized, 6 non-polarized
pair: 9 polarized, 8 non-polarized
pair: 11 polarized, 10 non-polarized
pair: 13 polarized, 12 non-polarized
pair: 15 polarized, 14 non-polarized
/run/media/vpc/kaffee/Computador_micro/PICRO-TUMOR-JULHO_2026/Fêmeas/A10/A10/A10_NDTiffStack.tif 20
pair: 1 polarized, 0 non-polarized
pair: 3 polarized, 2 non-polarized
pair: 5 polarized, 4 non-polarized
pair: 7 polarized, 6 non-polarized
pair: 9 polarized, 8 non-polarized
pair: 11 polarized, 10 non-polarized
pair: 13 polarized, 12 non-polarized
pair: 15 polarized, 14 non-polarized
pair: 17 polarized, 16 non-polarized
pair: 19 polarized, 18 non-polarized
/run/media/vpc/kaffee/Computador_micro/PICRO-TUMOR-JULHO_2026/Fêmeas/A17/A17/A17_NDTiffStack.tif 20
pair: 1 polarized, 0 non-polarized
pair: 3 polarized,